# 03 — Generator Quality

This notebook evaluates the conditional diffusion generator:
- Diffusion model architecture and UNet1D parameter count
- Cosine noise schedule visualisation
- Stylized fact tests (return distribution, volatility clustering, spread CDF, ...)
- LOB-Bench Wasserstein distances and discriminator accuracy
- Regime conditioning validation (low / normal / high volatility)

All cells run with dummy `np.random.randn(1000, 40)` data where real generated samples are absent.

In [ ]:
import matplotlib
import numpy as np
import pandas as pd

matplotlib.use('Agg')
import matplotlib.pyplot as plt
import torch

np.random.seed(42)
torch.manual_seed(42)
print('Dependencies loaded.')

## 1. Diffusion Model Architecture

The generator is a **conditional DDPM / DDIM** built from:

| Component | Role |
|-----------|------|
| `CosineNoiseSchedule` | Forward process betas; alpha-bar schedule |
| `SinusoidalTimestepEmbedding` | Timestep conditioning signal |
| `ConditioningModule` | Regime + predictor embedding conditioning |
| `UNet1D` | Denoising backbone; ~40M params with default config |
| `DiffusionModel` | Composes above; DDIM 10-step inference |

In [ ]:
from lob_forge.generator import CosineNoiseSchedule, UNet1D

# CosineNoiseSchedule
schedule = CosineNoiseSchedule(num_timesteps=1000, s=0.008)
print('CosineNoiseSchedule:')
print(f'  num_timesteps: {schedule.num_timesteps}')
print(f'  alpha_bar range: [{schedule.alphas_cumprod.min().item():.4f}, {schedule.alphas_cumprod.max().item():.4f}]')

# UNet1D (small config for demo)
unet = UNet1D(
    in_channels=40,
    d_model=64,
    channel_mults=(1, 2, 4),
    cond_dim=128,
)
unet_params = sum(p.numel() for p in unet.parameters())
print(f'\nUNet1D (small config) parameters: {unet_params:,}')

# Full config UNet1D
unet_full = UNet1D(
    in_channels=40,
    d_model=128,
    channel_mults=(1, 2, 4, 4),
    cond_dim=256,
)
unet_full_params = sum(p.numel() for p in unet_full.parameters())
print(f'UNet1D (full config, d_model=128) parameters:  {unet_full_params:,}')

In [ ]:
# Visualise the cosine alpha-bar schedule
alpha_bar = schedule.alphas_cumprod.numpy()  # shape (1000,)
timesteps = np.arange(len(alpha_bar))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(timesteps, alpha_bar, color='#4C72B0', linewidth=2)
ax1.set_xlabel('Timestep t')
ax1.set_ylabel('alpha_bar_t')
ax1.set_title('Cosine Schedule: Alpha-bar vs Timestep')
ax1.grid(True, alpha=0.3)

betas = 1.0 - alpha_bar[1:] / (alpha_bar[:-1] + 1e-12)
betas = np.clip(betas, 0, 0.999)
ax2.plot(np.arange(len(betas)), betas, color='#DD8452', linewidth=1.5)
ax2.set_xlabel('Timestep t')
ax2.set_ylabel('beta_t')
ax2.set_title('Noise Schedule: Beta vs Timestep')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('/tmp/noise_schedule.png', dpi=100)
plt.close()
print('Noise schedule plot saved.')


## 2. Stylized Fact Tests

LOB-Forge validates 6 stylized facts of financial markets:

| Test | What it checks |
|------|----------------|
| `return_distribution_test` | Fat tails (excess kurtosis > 0) |
| `volatility_clustering_test` | Autocorrelation of squared returns > 0 |
| `bid_ask_bounce_test` | Negative lag-1 return autocorrelation |
| `spread_cdf_test` | KS test on spread distributions |
| `book_shape_test` | KS test on per-level depth distributions |
| `market_impact_test` | Volume proxy vs price change correlation |

In [ ]:
import inspect

from lob_forge.evaluation.stylized_facts import run_all_stylized_tests

print('run_all_stylized_tests signature:')
print(' ', inspect.signature(run_all_stylized_tests))

In [ ]:
# Build realistic-ish dummy LOB arrays (40 columns, grouped layout)
# ask_price (0-9), ask_size (10-19), bid_price (20-29), bid_size (30-39)
N = 1000
base = 100.0

def make_lob_array(n, seed=0):
    rng = np.random.default_rng(seed)
    arr = np.zeros((n, 40), dtype=np.float32)
    mid_walk = base + rng.standard_normal(n).cumsum() * 0.01
    half_spread = np.abs(rng.standard_normal(n)) * 0.01 + 0.005
    for i in range(10):
        arr[:, i]      = mid_walk + half_spread * (1 + i * 0.5)   # ask prices
        arr[:, 10 + i] = rng.integers(10, 500, n).astype(float)   # ask sizes
        arr[:, 20 + i] = mid_walk - half_spread * (1 + i * 0.5)   # bid prices
        arr[:, 30 + i] = rng.integers(10, 500, n).astype(float)   # bid sizes
    return arr

real_lob = make_lob_array(N, seed=42)
synth_lob = make_lob_array(N, seed=123)  # slightly different distribution

print(f'Real LOB array shape:      {real_lob.shape}')
print(f'Synthetic LOB array shape: {synth_lob.shape}')

In [ ]:
# Run all 6 stylized fact tests
results = run_all_stylized_tests(real_lob, synth_lob)

rows = []
for test_name, result in results.items():
    rows.append({
        'Test': test_name,
        'Passed': result.get('passed', '—'),
        **{k: f'{v:.4f}' if isinstance(v, float) else v
           for k, v in result.items() if k != 'passed'},
    })

df_results = pd.DataFrame(rows)
print('Stylized Fact Test Results:')
print(df_results.to_string(index=False))

## 3. LOB-Bench Metrics

LOB-Bench computes two quantitative scores:
- **Wasserstein distance** (`wd_mean`) — distribution distance across LOB features
- **Discriminator accuracy** — MLP trained to distinguish real vs synthetic; closer to 0.5 = better

In [ ]:
import inspect

from lob_forge.evaluation.lob_bench import run_lob_bench

print('run_lob_bench signature:')
print(' ', inspect.signature(run_lob_bench))

In [ ]:
# Run LOB-Bench on dummy data
bench_results = run_lob_bench(real_lob, synth_lob)

print('LOB-Bench Results:')
for k, v in bench_results.items():
    if isinstance(v, float):
        print(f'  {k}: {v:.4f}')
    else:
        print(f'  {k}: {v}')

In [ ]:
# Plot Wasserstein distances per feature group
from lob_forge.evaluation.lob_bench import compute_wasserstein_metrics

wd_metrics = compute_wasserstein_metrics(real_lob, synth_lob)

feature_wds = {k: v for k, v in wd_metrics.items() if k.startswith('wd_feature_')}
if feature_wds:
    labels = [k.replace('wd_feature_', 'F') for k in feature_wds]
    values = list(feature_wds.values())

    fig, ax = plt.subplots(figsize=(12, 4))
    ax.bar(labels[:20], values[:20], color='#4C72B0', edgecolor='white')
    ax.set_xlabel('Feature index')
    ax.set_ylabel('Wasserstein Distance')
    ax.set_title('Per-Feature Wasserstein Distances (real vs synthetic)')
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.savefig('/tmp/wasserstein_distances.png', dpi=100)
    plt.close()
    print('Wasserstein distance plot saved.')
else:
    print('No per-feature WD keys — printing available keys:')
    for k, v in wd_metrics.items():
        print(f'  {k}: {v:.4f}' if isinstance(v, float) else f'  {k}: {v}')

## 4. Regime Conditioning

The generator conditions on 3 volatility regimes:
- **Regime 0** — Low volatility (calm market)
- **Regime 1** — Normal volatility
- **Regime 2** — High volatility (stressed market)

Regime labels come from realized volatility quantiles (0.33 and 0.67 thresholds).
`validate_regime_conditioning` checks that generated samples are statistically distinct across regimes.

In [ ]:
import inspect

from lob_forge.evaluation.regime_validation import validate_regime_conditioning

print('validate_regime_conditioning signature:')
print(' ', inspect.signature(validate_regime_conditioning))


In [ ]:
# Build per-regime dummy LOB data with distinct volatility levels
def make_regime_lob(n, vol_scale, seed=0):
    rng = np.random.default_rng(seed)
    arr = np.zeros((n, 40), dtype=np.float32)
    mid_walk = base + rng.standard_normal(n).cumsum() * vol_scale
    half_spread = np.abs(rng.standard_normal(n)) * vol_scale + 0.005
    for i in range(10):
        arr[:, i]      = mid_walk + half_spread * (1 + i * 0.5)
        arr[:, 10 + i] = rng.integers(10, 500, n).astype(float)
        arr[:, 20 + i] = mid_walk - half_spread * (1 + i * 0.5)
        arr[:, 30 + i] = rng.integers(10, 500, n).astype(float)
    return arr

regime_data = {
    0: make_regime_lob(N // 3, vol_scale=0.002, seed=10),  # low vol
    1: make_regime_lob(N // 3, vol_scale=0.01,  seed=11),  # normal
    2: make_regime_lob(N // 3, vol_scale=0.05,  seed=12),  # high vol
}

print('Per-regime data shapes:')
for regime, arr in regime_data.items():
    print(f'  Regime {regime}: {arr.shape}')

In [ ]:
# Run regime conditioning validation
# validate_regime_conditioning(real_by_regime, synthetic_by_regime)
# We use slightly perturbed regime data as 'synthetic' for demonstration
synth_by_regime = {
    0: make_regime_lob(N // 3, vol_scale=0.003, seed=20),
    1: make_regime_lob(N // 3, vol_scale=0.012, seed=21),
    2: make_regime_lob(N // 3, vol_scale=0.045, seed=22),
}

regime_val = validate_regime_conditioning(regime_data, synth_by_regime)

print('Regime Conditioning Validation:')
for k, v in regime_val.items():
    if isinstance(v, float):
        print('  ' + k + ':', round(v, 4))
    else:
        print('  ' + k + ':', v)


In [ ]:
# Plot mid-price return distributions per regime
fig, ax = plt.subplots(figsize=(10, 5))

colors = ['#55A868', '#4C72B0', '#C44E52']
labels = ['Regime 0 (Low Vol)', 'Regime 1 (Normal)', 'Regime 2 (High Vol)']

for regime, (color, label) in enumerate(zip(colors, labels, strict=False)):
    arr = regime_data[regime]
    mid = (arr[:, 0] + arr[:, 20]) / 2.0
    returns = np.diff(mid) / (mid[:-1] + 1e-12)
    ax.hist(returns, bins=60, alpha=0.6, color=color, label=label, density=True)

ax.set_xlabel('Mid-Price Return')
ax.set_ylabel('Density')
ax.set_title('Return Distribution by Regime (dummy data)')
ax.legend()

plt.tight_layout()
plt.savefig('/tmp/regime_returns.png', dpi=100)
plt.close()
print('Regime return distribution plot saved.')